In [1]:
import torch
from torch.utils.data import Dataset
from torchvision import datasets
from transformers import AutoModel
import torch.nn as nn
import os
from torch.utils.data import DataLoader, TensorDataset
from torchvision import transforms
from tqdm.notebook import tqdm
from icecream import ic
from PIL import Image
import random
import numpy as np
from torchvision.datasets import OxfordIIITPet
import timm
import torch.nn.functional as F
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup
from icecream import ic

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

/home/system/.local/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/system/.local/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias

In [4]:
model_large = AutoModel.from_pretrained('facebook/dinov2-large')
model_tiny = timm.create_model(
    'vit_tiny_patch16_224',
    pretrained=True,
    # num_classes=37
)

Loading weights:   0%|          | 0/439 [00:00<?, ?it/s]

In [5]:
model_large

Dinov2Model(
  (embeddings): Dinov2Embeddings(
    (patch_embeddings): Dinov2PatchEmbeddings(
      (projection): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14))
    )
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): Dinov2Encoder(
    (layer): ModuleList(
      (0-23): 24 x Dinov2Layer(
        (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
        (attention): Dinov2Attention(
          (attention): Dinov2SelfAttention(
            (query): Linear(in_features=1024, out_features=1024, bias=True)
            (key): Linear(in_features=1024, out_features=1024, bias=True)
            (value): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (output): Dinov2SelfOutput(
            (dense): Linear(in_features=1024, out_features=1024, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
        )
        (layer_scale1): Dinov2LayerScale()
        (drop_path): Identity()
        (norm2): LayerNorm((1024,),

In [3]:
model_large = AutoModel.from_pretrained('facebook/dinov2-large')
model_tiny = timm.create_model(
    'vit_tiny_patch16_224',
    pretrained=True,
    num_classes=37
)



class DistillationWrapper(nn.Module):
    def __init__(self, student, teacher_dim=1024):
        super().__init__()
        self.student = student
        self.proj = nn.Linear(student.embed_dim, teacher_dim)

    def forward(self, x):
        # student forward
        student_out = self.student.forward_features(x)  # [B, 197, 192]
        
        cls_token = student_out[:, 0]                  # [B, 192]
        projected = self.proj(cls_token)               # [B, 1024]
        
        return projected

student = DistillationWrapper(model_tiny)
ic(model_large.config.hidden_size)   # 1024
ic(model_tiny.embed_dim)             # 192
params_large = sum(p.numel() for p in model_large.parameters())
params_tiny = sum(p.numel() for p in student.parameters())

ic(params_large)
ic(params_tiny)
ic(params_tiny/params_large)

Loading weights:   0%|          | 0/439 [00:00<?, ?it/s]

ic| model_large.config.hidden_size: 1024
ic| model_tiny.embed_dim: 192
ic| params_large: 304368640
ic| params_tiny: 5729189
ic| params_tiny/params_large: 0.018823190851725066


0.018823190851725066

In [4]:
# compute mean over all parameters in model_large
total = 0.0
count = 0
for p in model_large.parameters():
    total += p.data.sum().item()      # sum of this tensor’s elements
    count += p.data.numel()           # number of elements

mean_val = total / count
print(mean_val)

-0.00024278295886696698


In [5]:


class OxfordPetDataset(Dataset):
    def __init__(self, root_dir, split="trainval", transform=None):
        """
        root_dir: path to 'oxford-iiit-pet'
        split: 'trainval' or 'test'
        transform: torchvision transforms
        """
        self.root_dir = root_dir
        self.images_dir = os.path.join(root_dir, "images")
        self.annotations_file = os.path.join(root_dir, "annotations", f"{split}.txt")
        # self.transform = transform
        if transform is None:
            self.transform = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                     std=[0.229, 0.224, 0.225])
            ])
        else:
            self.transform = transform

        self.samples = []
        self._load_annotations()
        ic(len(self.samples))

    def _load_annotations(self):
        with open(self.annotations_file, "r") as f:
            for line in tqdm(f):
                parts = line.strip().split()
                image_name = parts[0] + ".jpg"
                label = int(parts[1]) - 1  # convert to 0-based index
                
                self.samples.append((image_name, label))
    
    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_name, label = self.samples[idx]

        image_path = os.path.join(self.images_dir, image_name)
        image = Image.open(image_path).convert("RGB")
        image = image.resize((224, 224))

        if self.transform:
            image = self.transform(image)

        return image, label


train_dataset = OxfordPetDataset(root_dir="/media/system/ZERBUIS_EXT_STOR/dynamic_slam/experiments/data/oxford-iiit-pet", split="trainval")
train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=20,      # How many samples per batch
    shuffle=True)

test_dataset = OxfordPetDataset(root_dir="/media/system/ZERBUIS_EXT_STOR/dynamic_slam/experiments/data/oxford-iiit-pet", split="test")
test_dataloader = DataLoader(
    dataset=test_dataset,
    batch_size=16,      # How many samples per batch
    shuffle=False)


3680it [00:00, 2846217.72it/s]
ic| len(self.samples): 3680
3669it [00:00, 2402263.72it/s]
ic| len(self.samples): 3669


In [6]:
# BATCH_SIZE = 32
# LR = 5e-5
# EPOCHS = 10
# WEIGHT_DECAY = 0.05
# WARMUP_RATIO = 0.05

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# ic(device)
# student = student.to(device)

# optimizer = torch.optim.AdamW(
#     student.parameters(),
#     lr=LR,
#     weight_decay=WEIGHT_DECAY
# )

# num_training_steps = EPOCHS * len(train_dataloader)
# num_warmup_steps = int(WARMUP_RATIO * num_training_steps)

# scheduler = get_cosine_schedule_with_warmup(
#     optimizer,
#     num_warmup_steps=num_warmup_steps,
#     num_training_steps=num_training_steps
# )

# criterion = nn.MSELoss()


# output_dir = 'student_weights'
# os.makedirs(output_dir, exist_ok=True)
# train_losses = []
# test_losses = []
# import time
# model_large = model_large.to(device)


# for epoch in tqdm(range(EPOCHS)):
#     # ---------------- TRAIN ----------------
#     student.train()
#     batch_loss = 0
#     for batch in tqdm(train_dataloader):
#         images, labels = batch
#         images = images.to(device)
#         target_output = model_large(images)
#         target_output = target_output.last_hidden_state
#         # target_output = target_output.to(device)
#         target_output = target_output[:, 0]
#         output = student(images)
#         loss = criterion(output, target_output)
#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()
#         scheduler.step()
#         batch_loss += loss.item()
#     train_losses.append(batch_loss / len(train_dataloader))
#     print(f"Epoch {epoch+1}/{EPOCHS}, Train Loss: {train_losses[-1]:.4f}")
    
#     with torch.no_grad():
#         student.eval()
#         batch_loss = 0
#         for batch in test_dataloader:
#             images, labels = batch
#             images = images.to(device)
#             target_output = model_large(images)
#             target_output = target_output.last_hidden_state
#             target_output = target_output[:, 0]
#             output = student(images)
#             loss = criterion(output, target_output)
#             batch_loss += loss.item()
#         test_losses.append(batch_loss / len(test_dataloader))
#         print(f"Epoch {epoch+1}/{EPOCHS}, Test Loss: {test_losses[-1]:.4f}")
#     torch.save(student.state_dict(), f"{output_dir}/student_epoch_{epoch+1}.pt")

Cosine = 0.9 → loss ≈ 0.2 \\
Cosine = 0.8 → loss ≈ 0.4  \\
Cosine = 0.5 → loss ≈ 1 \\


In [7]:

BATCH_SIZE = 32
LR = 1e-4
EPOCHS = 20
WEIGHT_DECAY = 0.05
WARMUP_RATIO = 0.05

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

student = student.to(device)
model_large = model_large.to(device)

model_large.eval()
for p in model_large.parameters():
    p.requires_grad = False

optimizer = AdamW(
    student.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

num_training_steps = EPOCHS * len(train_dataloader)
num_warmup_steps = int(WARMUP_RATIO * num_training_steps)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

output_dir = "student_weights"
os.makedirs(output_dir, exist_ok=True)

train_losses = []
test_losses = []
train_cosines = []
test_cosines = []

for epoch in range(EPOCHS):

    # ---------------- TRAIN ----------------
    student.train()
    total_loss = 0
    total_cos = 0

    for images, _ in tqdm(train_dataloader, desc=f"Train Epoch {epoch+1}"):

        images = images.to(device)

        with torch.no_grad():
            teacher_output = model_large(images).last_hidden_state[:, 0]

        student_output = student(images)

        teacher_norm = F.normalize(teacher_output, dim=-1)
        student_norm = F.normalize(student_output, dim=-1)

        loss = 2 - 2 * (student_norm * teacher_norm).sum(dim=-1).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        # Track cosine similarity
        cos_sim = F.cosine_similarity(student_norm, teacher_norm, dim=-1).mean().item()
        total_cos += cos_sim

    avg_train_loss = total_loss / len(train_dataloader)
    avg_train_cos = total_cos / len(train_dataloader)

    train_losses.append(avg_train_loss)
    train_cosines.append(avg_train_cos)

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Train Cosine Similarity: {avg_train_cos:.4f}")

    student.eval()
    total_loss = 0
    total_cos = 0

    with torch.no_grad():
        for images, _ in tqdm(test_dataloader, desc="Validation"):

            images = images.to(device)

            teacher_output = model_large(images).last_hidden_state[:, 0]
            student_output = student(images)

            teacher_norm = F.normalize(teacher_output, dim=-1)
            student_norm = F.normalize(student_output, dim=-1)

            loss = 2 - 2 * (student_norm * teacher_norm).sum(dim=-1).mean()

            total_loss += loss.item()

            cos_sim = F.cosine_similarity(student_norm, teacher_norm, dim=-1).mean().item()
            total_cos += cos_sim

    avg_test_loss = total_loss / len(test_dataloader)
    avg_test_cos = total_cos / len(test_dataloader)

    test_losses.append(avg_test_loss)
    test_cosines.append(avg_test_cos)

    print(f"Val Loss: {avg_test_loss:.4f}")
    print(f"Val Cosine Similarity: {avg_test_cos:.4f}")

    # Save checkpoint
    torch.save(student.state_dict(),
               f"{output_dir}/student_epoch_{epoch+1}.pt")

print("\nTraining Complete.")

Device: cuda


Train Epoch 1: 100%|██████████| 184/184 [00:43<00:00,  4.20it/s]



Epoch 1/20
Train Loss: 1.6670
Train Cosine Similarity: 0.1665


Validation: 100%|██████████| 230/230 [00:38<00:00,  5.90it/s]


Val Loss: 1.3306
Val Cosine Similarity: 0.3347


...


Epoch 14/20
Train Loss: 0.3748
Train Cosine Similarity: 0.8126


Validation: 100%|██████████| 230/230 [00:38<00:00,  5.91it/s]


Val Loss: 0.5510
Val Cosine Similarity: 0.7245


Train Epoch 15:  65%|██████▍   | 119/184 [00:28<00:15,  4.14it/s]


KeyboardInterrupt: 